In [1]:
#!pip install geopandas pandas shapely

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Define the base directory (Uncomment the active path)
base_path = 'E:\\Projetos\\ABM-WP' # Currently active path

# Input paths
shapefile_path = os.path.join(base_path, 'includes', 'maps', 'itapua', 'Mapa_Itapua.shp')
consumers_csv_path = os.path.join(base_path, 'includes', 'Tabela_consumidores_Itapua_convertida.csv')

# Output path
output_csv_path = os.path.join(base_path, 'includes', 'Tabela_consumidores_Itapua_com_setor.csv')

# ==========================================
# 2. Load Census Tracts (Shapefile)
# ==========================================

# Load the shapefile
census_tracts = gpd.read_file(shapefile_path)

# Select only the Sector ID (CD_SETOR) and geometry columns
census_tracts = census_tracts[['CD_SETOR', 'geometry']]

# Display the first few rows to verify loading
print("Census Tracts loaded:")
print(census_tracts.head())

# ==========================================
# 3. Load Consumer Data (CSV)
# ==========================================

# Load the CSV file containing supply points
# separator is set to ';' as per original file structure
consumers_df = pd.read_csv(consumers_csv_path, sep=';')

print("\nConsumer Data loaded:")
print(consumers_df.head())

# ==========================================
# 4. Convert to GeoDataFrame
# ==========================================

# Create Point geometries from Longitude and Latitude columns
# Note: Ensure the columns 'LONG_GEO' and 'LAT_GEO' exist in your CSV
geometry = [Point(xy) for xy in zip(consumers_df['LONG_GEO'], consumers_df['LAT_GEO'])]

# Initialize GeoDataFrame
# EPSG:4326 is the standard WGS84 coordinate system (Lat/Lon)
consumers_gdf = gpd.GeoDataFrame(consumers_df, geometry=geometry, crs="EPSG:4326")

# ==========================================
# 5. Spatial Join
# ==========================================

# Reproject consumer points to match the CRS of the census tracts shapefile
consumers_gdf = consumers_gdf.to_crs(census_tracts.crs)

# Perform Spatial Join (Left Join)
# 'predicate="within"' checks if the point falls inside the polygon
spatial_join_result = gpd.sjoin(consumers_gdf, census_tracts, how="left", predicate="within")

# Drop unnecessary columns generated by the join
spatial_join_result.drop(columns=['geometry', 'index_right'], inplace=True)

# Remove points that fell outside the target census sectors (if any)
spatial_join_result.dropna(subset=['CD_SETOR'], inplace=True)

# ==========================================
# 6. Save Results
# ==========================================

# Save the enriched dataframe to a new CSV file
spatial_join_result.to_csv(output_csv_path, index=False)

print(f"\nProcessing complete. File saved to: {output_csv_path}")
print("Sample output:")
print(spatial_join_result.head())

Census Tracts loaded:
          CD_SETOR                                           geometry
0  292740805090020  POLYGON ((-38.36855 -12.94796, -38.36859 -12.9...
1  292740805090021  POLYGON ((-38.36738 -12.94851, -38.36739 -12.9...
2  292740805090022  POLYGON ((-38.36442 -12.94586, -38.36457 -12.9...
3  292740805090023  POLYGON ((-38.36433 -12.94755, -38.36433 -12.9...
4  292740805090024  POLYGON ((-38.36165 -12.94856, -38.36168 -12.9...

Consumer Data loaded:
                                        SK_MATRICULA NM_LOCALIDADE  \
0  6E60FF997A2A7FD7665AC40BAD9BE286D7C47A1C3521CE...  SALVADOR UMB   
1  E6195B23791D174F7C612B6324CACDED0CBDC2283FC728...  SALVADOR UMB   
2  0C2ED22E52F7B889C5DCB715B44E40C0CF4E25A77F1A17...  SALVADOR UMB   
3  6D38EE4CEC85F41B626000179F50D3599EEFA9B20AC328...  SALVADOR UMB   
4  0420BAE41A9CA4FBA6CE1F1E120C73D05448330C834D72...  SALVADOR UMB   

  NM_CATEGORIATARIFARIA NM_SITUACAO_IMOVEL  NN_MORADORES  ST_PISCINA  \
0           RESIDENCIAL           HABITADO